# Visual diagnostics for the Telco rule set

Four publication-grade figures over the Telco rule set mined in
`01_end_to_end_telco.ipynb`:

1. **Churn-rate panel** — base rates per value of each flexible attribute,
   overlaid with the dataset-wide rate. Sets context for why the action-rules
   approach is interesting on this data.
2. **Bootstrap distribution** for the top rule by realistic rule gain, with
   the analytic 95 % CI overlaid for comparison.
3. **Forest plot** with category markers (Accept / Uncertain / Reject) at a
   break-even threshold of 150 currency units.
4. **Bootstrap vs analytic CI-width scatter** — how the two methods compare
   per rule, coloured by rule support.

All four figures are saved to `notebooks/telco_tour/outputs/` as PDFs at
publication resolution. The code prefers shape/shade differentiation over
hue so the PDFs reproduce on a black-and-white printer.


In [ ]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

# Locate repo root from the notebook's working directory.
REPO_ROOT = Path.cwd()
while not (REPO_ROOT / 'pyproject.toml').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if not (REPO_ROOT / 'pyproject.toml').exists():
    raise RuntimeError('Repository root with pyproject.toml not found.')

TELCO_CSV = REPO_ROOT / 'notebooks' / 'data' / 'telco.csv'
OUT_DIR = REPO_ROOT / 'notebooks' / 'telco_tour' / 'outputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings('ignore')
print(f'Repo root: {REPO_ROOT}')
print(f'Telco CSV: {TELCO_CSV.relative_to(REPO_ROOT)}')
print(f'Out dir:   {OUT_DIR.relative_to(REPO_ROOT)}')


## Matplotlib styling

Grayscale settings shared by every figure. Marker shape, line style, and fill
shade do the differentiating; nothing is colour-encoded.


In [ ]:
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 9,
    'axes.titlesize': 9.5,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'axes.linewidth': 0.6,
    'lines.linewidth': 0.9,
    'patch.linewidth': 0.5,
    'xtick.major.width': 0.5,
    'ytick.major.width': 0.5,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.edgecolor': 'black',
    'axes.facecolor': 'white',
    'savefig.facecolor': 'white',
    'figure.facecolor': 'white',
    'grid.color': 'black',
    'grid.alpha': 0.15,
    'grid.linewidth': 0.4,
})


## Load Telco and (re)mine the rule set


In [ ]:
from action_rules import ActionRules

STABLE = ['gender', 'SeniorCitizen', 'Partner']
FLEXIBLE = [
    'PhoneService', 'InternetService', 'OnlineSecurity',
    'DeviceProtection', 'TechSupport', 'StreamingTV',
]
TARGET = 'Churn'
UNDESIRED, DESIRED = 'Yes', 'No'

INTRINSIC = {
    ('Churn', 'No'): 400.0, ('Churn', 'Yes'): 0.0,
    ('OnlineSecurity',   'Yes'): -10.0, ('OnlineSecurity',   'No'): 0.0,
    ('DeviceProtection', 'Yes'):  -8.0, ('DeviceProtection', 'No'): 0.0,
    ('TechSupport',      'Yes'): -12.0, ('TechSupport',      'No'): 0.0,
    ('StreamingTV',      'Yes'):  -5.0, ('StreamingTV',      'No'): 0.0,
}
TRANSITION = {
    ('OnlineSecurity',   'No', 'Yes'): -5.0,
    ('DeviceProtection', 'No', 'Yes'): -4.0,
    ('TechSupport',      'No', 'Yes'): -6.0,
    ('StreamingTV',      'No', 'Yes'): -3.0,
}

df = pd.read_csv(TELCO_CSV, sep=';')
df['SeniorCitizen'] = df['SeniorCitizen'].astype(str)

ar = ActionRules(
    min_stable_attributes=2,
    min_flexible_attributes=1,
    min_undesired_support=220,
    min_desired_support=110,
    min_undesired_confidence=0.6,
    min_desired_confidence=0.6,
    intrinsic_utility_table=INTRINSIC,
    transition_utility_table=TRANSITION,
)
ar.fit(
    data=df,
    stable_attributes=STABLE,
    flexible_attributes=FLEXIBLE,
    target=TARGET,
    target_undesired_state=UNDESIRED,
    target_desired_state=DESIRED,
)
print(f'mined {len(ar.output.action_rules)} rules on {len(df):,} customers')


## Figure 1 — Churn rate per flexible-attribute value

Shows which attribute values are over-represented among churners; the dashed
line is the overall churn rate. If every value sat at the dashed line the
attribute would carry no signal.


In [ ]:
overall = (df['Churn'] == 'Yes').mean()
fig, axes = plt.subplots(2, 3, figsize=(8.4, 4.2), sharey=True)
for ax, col in zip(axes.flat, FLEXIBLE):
    rates = df.groupby(col, sort=False)['Churn'].apply(lambda s: (s == 'Yes').mean())
    rates = rates.sort_values(ascending=False)
    ax.bar(range(len(rates)), rates.values, color='0.55', edgecolor='black', linewidth=0.5)
    ax.set_xticks(range(len(rates)))
    ax.set_xticklabels(rates.index, rotation=30, ha='right', fontsize=7)
    ax.axhline(overall, color='black', linestyle='--', linewidth=0.7,
               label=f'overall {overall:.2f}' if col == FLEXIBLE[0] else None)
    ax.set_title(col, fontsize=9)
    ax.set_ylim(0, 0.55)
    ax.grid(axis='y', linestyle=':', linewidth=0.4, alpha=0.5)
for ax in axes[:, 0]:
    ax.set_ylabel('P(Churn = Yes)')
axes[0, 0].legend(loc='upper right', frameon=False)
fig.tight_layout()
fig.savefig(OUT_DIR / 'fig1_churn_rate_panel.pdf', bbox_inches='tight')
plt.show()


## Bootstrap + analytic CIs (needed for the next three figures)

We compute both because the visual comparison is the headline of Figs. 2 and 4.


In [ ]:
threshold = 150.0
boot = ar.confidence_intervals(
    data=df,
    method='bootstrap',
    bootstrap_type='percentile',
    n_bootstrap=1000,
    random_state=42,
    metric='realistic_rule_gain',
    threshold=threshold,
)
ana = ar.confidence_intervals(
    data=df,
    method='analytic',
    analytic_type='auto',
    metric='realistic_rule_gain',
    threshold=threshold,
)

boot_sorted = sorted(boot, key=lambda r: -(r.realistic_rule_gain_point or float('-inf')))
top = boot_sorted[0]
top_ana = next(r for r in ana if r.rule_index == top.rule_index)
print(f'top rule by gain point: rule {top.rule_index}, gain ~ {top.realistic_rule_gain_point:.1f}')


## Figure 2 — Bootstrap distribution + analytic overlay (top rule)

Histogram = the bootstrap sampling distribution of the realistic rule gain.
Dashed lines + hatched span = bootstrap 95 % CI. Dotted lines = analytic
auto Wald/Wilson 95 % CI. Solid line = point estimate. Tight agreement
between the two intervals on this rule is reassuring.


In [ ]:
fig, ax = plt.subplots(figsize=(6.2, 3.4))
samples = top.samples_gain
ax.hist(samples, bins=40, density=True, facecolor='0.80', edgecolor='black', linewidth=0.4)
ax.axvspan(
    top.realistic_rule_gain_ci_lower, top.realistic_rule_gain_ci_upper,
    facecolor='none', edgecolor='black', hatch='////', alpha=0.22, linewidth=0.0,
)
ax.axvline(top.realistic_rule_gain_ci_lower, color='black', linestyle='--', linewidth=0.9)
ax.axvline(top.realistic_rule_gain_ci_upper, color='black', linestyle='--', linewidth=0.9)
ax.axvline(top.realistic_rule_gain_point,    color='black', linestyle='-',  linewidth=1.2)
ax.axvline(top_ana.realistic_rule_gain_ci_lower, color='black', linestyle=':', linewidth=1.2)
ax.axvline(top_ana.realistic_rule_gain_ci_upper, color='black', linestyle=':', linewidth=1.2)
ax.set_xlabel(r'Realistic rule gain $u(r_{1\to 2})$')
ax.set_ylabel('Density')

boot_lo, boot_hi = top.realistic_rule_gain_ci_lower, top.realistic_rule_gain_ci_upper
ana_lo, ana_hi   = top_ana.realistic_rule_gain_ci_lower, top_ana.realistic_rule_gain_ci_upper
legend_handles = [
    Patch(facecolor='0.80', edgecolor='black', label=r'bootstrap samples ($B = 1000$)'),
    Line2D([], [], color='black', linestyle='-',  linewidth=1.2,
           label=r'point est. $\hat u = ' + f'{top.realistic_rule_gain_point:.1f}$'),
    Line2D([], [], color='black', linestyle='--', linewidth=0.9,
           label=r'bootstrap 95% CI: ' + f'[{boot_lo:.1f}, {boot_hi:.1f}]'),
    Line2D([], [], color='black', linestyle=':',  linewidth=1.2,
           label=r'analytic-auto 95% CI: ' + f'[{ana_lo:.1f}, {ana_hi:.1f}]'),
]
leg = ax.legend(handles=legend_handles, loc='upper left', frameon=True, framealpha=1.0,
                facecolor='white', edgecolor='black', fontsize=7.5)
leg.get_frame().set_linewidth(0.4)
ax.grid(axis='y', linestyle=':', linewidth=0.4, alpha=0.5)
fig.tight_layout()
fig.savefig(OUT_DIR / 'fig2_bootstrap_hist.pdf', bbox_inches='tight')
plt.show()


## Figure 3 — Forest plot with category markers

Each rule is plotted at its analytic-auto point estimate with horizontal
error bars for the 95 % CI. Marker shape encodes the threshold-based
verdict at `tau = 150`:

- **Filled circle** = Accept (entire CI above tau).
- **Open diamond** = Uncertain (CI straddles tau).
- **Cross** = Reject (entire CI below tau).


In [ ]:
points = np.array([r.realistic_rule_gain_point for r in ana])
lows   = np.array([r.realistic_rule_gain_ci_lower for r in ana])
highs  = np.array([r.realistic_rule_gain_ci_upper for r in ana])
cats   = [str(r.category).split('.')[-1].lower() for r in ana]
idxs   = [r.rule_index for r in ana]
order = np.argsort(points)
points, lows, highs = points[order], lows[order], highs[order]
cats = [cats[i] for i in order]
idxs = [idxs[i] for i in order]
n = len(points)

fig, ax = plt.subplots(figsize=(6.6, 0.22 * n + 0.6))
marker_for = {'accept': 'o', 'uncertain': 'D', 'reject': 'x'}
fill_for   = {'accept': 'black', 'uncertain': 'white', 'reject': 'black'}
for i, (p, lo, hi, cat) in enumerate(zip(points, lows, highs, cats)):
    ax.errorbar(
        p, i,
        xerr=[[p - lo], [hi - p]],
        fmt=marker_for[cat],
        mfc=fill_for[cat],
        mec='black', ecolor='black',
        elinewidth=0.6, capsize=2.0,
        markersize=5.0, markeredgewidth=0.7,
    )
ax.axvline(threshold, color='black', linestyle='--', linewidth=0.7)
ax.set_yticks(range(n))
ax.set_yticklabels([f'r{idx}' for idx in idxs], fontsize=7)
ax.set_xlabel(r'Realistic rule gain (analytic-auto $95\%$ CI)')
ax.set_ylabel('rule index')
ax.grid(axis='x', linestyle=':', linewidth=0.4, alpha=0.5)
ax.legend(
    handles=[
        Line2D([], [], marker='o', mfc='black', mec='black', linestyle='', markersize=5, label='Accept'),
        Line2D([], [], marker='D', mfc='white', mec='black', linestyle='', markersize=5, label='Uncertain'),
        Line2D([], [], marker='x', mfc='black', mec='black', linestyle='', markersize=5, label='Reject'),
        Line2D([], [], color='black', linestyle='--', linewidth=0.7, label=f'threshold $\\tau = {threshold:g}$'),
    ],
    loc='lower right', frameon=False,
)
fig.tight_layout()
fig.savefig(OUT_DIR / 'fig3_forest_plot.pdf', bbox_inches='tight')
plt.show()


## Figure 4 — Bootstrap vs analytic CI-width scatter

Each point is one rule. The dashed line is `y = x`. Points above the line are
rules where the analytic interval is wider than the bootstrap interval; below,
the bootstrap is wider. Colour shade encodes rule support, so we can see
whether divergences cluster on the low-support side (small N → normal
approximation gets shaky).


In [ ]:
boot_w = np.array([(r.realistic_rule_gain_ci_upper - r.realistic_rule_gain_ci_lower) for r in boot])
ana_w  = np.array([(r.realistic_rule_gain_ci_upper - r.realistic_rule_gain_ci_lower) for r in ana])
sup = np.array([r.support for r in ana])

fig, ax = plt.subplots(figsize=(5.2, 4.2))
sc = ax.scatter(boot_w, ana_w, c=sup, cmap='Greys',
                vmin=sup.min() * 0.7, vmax=sup.max() * 1.05,
                s=42, edgecolor='black', linewidth=0.5)
lo = min(boot_w.min(), ana_w.min()) * 0.92
hi = max(boot_w.max(), ana_w.max()) * 1.05
ax.plot([lo, hi], [lo, hi], color='black', linestyle='--', linewidth=0.7, label=r'$y=x$')
ax.set_xlabel('Bootstrap CI width')
ax.set_ylabel('Analytic-auto CI width')
ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)
ax.grid(linestyle=':', linewidth=0.4, alpha=0.5)
cbar = fig.colorbar(sc, ax=ax, fraction=0.045, pad=0.04)
cbar.outline.set_linewidth(0.4)
cbar.set_label(r'rule support $n_u$')
ax.legend(loc='upper left', frameon=False)
fig.tight_layout()
fig.savefig(OUT_DIR / 'fig4_bootstrap_vs_analytic.pdf', bbox_inches='tight')
plt.show()


## Outputs

Four PDFs land in `notebooks/telco_tour/outputs/`:

- `fig1_churn_rate_panel.pdf`
- `fig2_bootstrap_hist.pdf`
- `fig3_forest_plot.pdf`
- `fig4_bootstrap_vs_analytic.pdf`

All four are reproducible byte-by-byte from a clean kernel because the
bootstrap is seeded (`random_state=42`).
